# Lab 5 · Phân khúc giá & host chuyên nghiệp (groupby, transform, merge)

**Lập trình xử lý dữ liệu (LTXLDL) · 2627-1 · Giờ thực hành tuần 5**

> 💡 File → **Save a copy in Drive** trước khi sửa.

Notebook demo buổi 5 phân tích theo **quận**. Lab này nhìn thị trường theo hai chiều khác:
**phân khúc giá** (rẻ / trung / cao) và **kiểu host** (cá nhân / chuyên nghiệp) — dùng trọn bộ
map → groupby/agg → transform → pivot_table → merge.

## Cách làm việc trong buổi lab

- Bài tập được chia bước; mỗi bước có ô `TODO` và phần kiểm tra `assert` — chạy qua hết
  `assert` nghĩa là bạn làm đúng.
- Phần khởi động và bài có hướng dẫn: bạn nên **tự gõ, không dùng AI** — các bài kiểm tra
  định kỳ 🔒 ở giờ lý thuyết đo đúng các kỹ năng này.
- Bài tự làm ở cuối được gắn nhãn 🔓: bạn được dùng AI, kèm trách nhiệm khai báo
  theo chính sách AI của môn.
- Bạn kẹt quá 3 phút ở một bước: gọi trợ giảng.

## Mục tiêu

Sau buổi lab, bạn:

1. Dùng `map` với **hàm** để sinh cột phân loại từ cột số.
2. `groupby` + `agg` đặt tên trên nhiều cột; đọc kết quả thành nhận xét.
3. Dùng `transform` để gắn thông tin nhóm về từng dòng.
4. `pivot_table` hai chiều và `merge` bảng tra cứu có `validate`.

## Phần 0 · Khởi động (~10 phút)

In [ ]:
import pandas as pd

# W1 — loc theo nhãn, iloc theo vị trí
s = pd.Series([100, 200, 300], index=[5, 0, 2])

# TODO: điền hai giá trị
theo_nhan = ...      # phần tử mang NHÃN 2
theo_vi_tri = ...    # phần tử ở VỊ TRÍ 2 (thứ ba)

# --- Ô kiểm tra ---
assert theo_nhan == 300 and theo_vi_tri == 300 or True  # xem giải thích dưới
assert s.loc[0] == 200 and s.iloc[0] == 100
assert theo_nhan == s.loc[2] and theo_vi_tri == s.iloc[2]
print("W1 ổn: loc[2] =", theo_nhan, "· iloc[2] =", theo_vi_tri)

In [ ]:
# W2 — alignment: phép toán tự khớp theo index
gia_cu = pd.Series({"A": 100, "B": 200, "C": 50})
gia_moi = pd.Series({"B": 220, "A": 90, "D": 70})

# TODO: tính thay đổi tuyệt đối (gia_moi - gia_cu) và đếm số NaN trong kết quả
thay_doi = ...
so_nan = ...

# --- Ô kiểm tra ---
assert thay_doi["A"] == -10 and thay_doi["B"] == 20
assert so_nan == 2
print("W2 ổn: pandas khớp theo NHÃN (A với A, B với B) — C và D lệch nhau thành NaN.")

## Phần 1 · Hai chiều nhìn mới về thị trường (~55 phút)

In [ ]:
URL = ("https://data.insideairbnb.com/chile/rm/santiago/"
       "2026-06-29/visualisations/listings.csv")
df = pd.read_csv(URL)
len(df)

### Bước 1 · Sinh cột phân khúc bằng map với hàm

Chia giá thành 3 phân khúc: `re` (< 30.000), `trung` (30.000–99.999), `cao` (≥ 100.000);
giá thiếu → `None`. Đây là **nấc 2** trong ba nấc biến đổi: logic có rẽ nhánh, vector hoá
thuần không diễn đạt gọn được — map với hàm là đúng chỗ.

In [ ]:
def phan_khuc(gia):
    # TODO: viết thân hàm theo mô tả trên (nhớ xử lý NaN bằng pd.isna trước)
    ...

df["phan_khuc"] = df["price"].map(phan_khuc)
dem = df["phan_khuc"].value_counts(dropna=False)

# --- Ô kiểm tra ---
assert dem["trung"] == 12325 and dem["cao"] == 3703 and dem["re"] == 1660
print(dem)

### Bước 2 · agg đặt tên — chân dung 3 phân khúc

`number_of_reviews_ltm` = số review 12 tháng qua — proxy thô cho "mức độ được đặt".

In [ ]:
# TODO: groupby phan_khuc, agg 3 cột đặt tên:
#   so_phong   = ("id", "size")
#   gia_trung_vi = ("price", "median")
#   review_nam_tb = ("number_of_reviews_ltm", "mean")
tk = ...

# --- Ô kiểm tra ---
assert tk.loc["trung", "so_phong"] == 12325
assert tk.loc["cao", "gia_trung_vi"] == 145318.0
assert round(tk.loc["trung", "review_nam_tb"], 1) == 15.4
tk.round(1)

Đọc bảng: phân khúc **trung** vừa đông nhất vừa được đặt nhiều nhất (15,4 review/năm) —
phân khúc "cao" giá gấp gần 3 lần nhưng khách thưa hơn. Một bảng agg 3 dòng đã là một
nhận xét thị trường.

### Bước 3 · Host chuyên nghiệp

Cột `calculated_host_listings_count` = host của phòng này đang có bao nhiêu listing.
Gọi host có **≥ 5 listing** là "chuyên nghiệp".

In [ ]:
# TODO: tạo cột bool df["chuyen"]; tính tỷ lệ phòng thuộc host chuyên nghiệp
df["chuyen"] = ...
ty_le_chuyen = ...

# TODO: bảng so sánh 2 nhóm chuyen (groupby "chuyen"), 2 cột đặt tên:
#   gia_trung_vi=("price", "median") · review_nam_tb=("number_of_reviews_ltm", "mean")
so_sanh = ...

# --- Ô kiểm tra ---
assert round(ty_le_chuyen, 3) == 0.358
assert so_sanh.loc[True, "gia_trung_vi"] == 65078.0
assert round(so_sanh.loc[False, "review_nam_tb"], 1) == 12.7
so_sanh.round(1)

36% thị trường nằm trong tay host chuyên nghiệp — nhóm này đặt giá cao hơn (~17%)
mà vẫn được đặt nhiều hơn. Câu hỏi vấn đáp tiềm năng: *kết luận "chuyên nghiệp làm ăn
tốt hơn" đã đứng vững chưa, hay còn biến ẩn?* (gợi ý: họ tập trung ở quận nào?)

### Bước 4 · transform — gắn cỡ quận về từng dòng

`transform` trả kết quả **đúng độ dài bảng gốc**: dùng nó để mỗi dòng biết quận của mình
to cỡ nào, rồi lọc bỏ các quận quá nhỏ trước khi so sánh.

In [ ]:
# TODO: tạo cột n_quan = cỡ quận (transform "size" trên groupby neighbourhood)
df["n_quan"] = ...
# TODO: lọc df_lon = các dòng thuộc quận có >= 300 phòng
df_lon = ...

# --- Ô kiểm tra ---
assert len(df_lon) == 16850
assert df_lon["neighbourhood"].nunique() == 8
print(f"{len(df_lon):,} phòng thuộc {df_lon['neighbourhood'].nunique()} quận đủ lớn.")

### Bước 5 · pivot_table — phân khúc × kiểu host

In [ ]:
# TODO: pivot_table trên df: index=phan_khuc, columns=chuyen,
#       values=number_of_reviews_ltm, aggfunc="mean"
pv = ...

# --- Ô kiểm tra ---
assert round(pv.loc["trung", True], 1) == 16.5
assert round(pv.loc["cao", False], 1) == 9.5
pv.round(1)

Đọc bảng chéo: ở **mọi** phân khúc, host chuyên nghiệp đều được đặt nhiều hơn —
khoảng cách rõ nhất ở phân khúc cao (13,5 vs 9,5). Kết luận Bước 3 "sống sót" qua một
lần cắt lớp — chưa phải bằng chứng nhân quả, nhưng vững hơn một bảng gộp.

### Bước 6 · merge bảng tra cứu + validate

Giả sử công ty áp **phí dịch vụ theo phân khúc**: rẻ 10%, trung 13%, cao 15%.
Bảng tra cứu 3 dòng — merge vào 18.534 dòng.

In [ ]:
phi = pd.DataFrame({"phan_khuc": ["re", "trung", "cao"],
                    "phi_dich_vu": [0.10, 0.13, 0.15]})

# TODO: merge df với phi theo phan_khuc, how="left", validate="m:1"
m = ...

# Thói quen sau merge: số dòng + độ khớp
# TODO: đếm số dòng của m và số NaN của cột phi_dich_vu
so_dong_m = ...
so_nan_phi = ...

# --- Ô kiểm tra ---
assert so_dong_m == 18534, "merge làm đổi số dòng?!"
assert so_nan_phi == 846, "NaN đúng bằng số phòng không có phân khúc (không giá)"
m["gia_tong"] = m["price"] * (1 + m["phi_dich_vu"])
gia_tong_tv = m.groupby("phan_khuc")["gia_tong"].median().round(0)
assert gia_tong_tv["trung"] == 61897.0
gia_tong_tv

846 dòng NaN phí **không phải lỗi merge** — đó là các phòng không giá nên không có
phân khúc. Biết giải thích từng NaN sau merge chính là "độ khớp" mà buổi 5 yêu cầu kiểm.

## Phần 2 · Bài tự làm 🔓 (làm xong sớm / về nhà)

Được dùng AI theo quy trình 5 bước; ghi lại prompt + cách kiểm chứng.

### Tự làm 1 · Quán quân từng quận

Với `df_lon` (8 quận lớn): dùng `transform("max")` trên giá theo quận để tìm **phòng đắt
nhất của từng quận** (so `price` với max quận). In bảng name / neighbourhood / price.
Kiểm chứng: quán quân của quận Santiago phải là căn 97.000.045 CLP quen thuộc.

### Tự làm 2 · Cơ cấu phân khúc theo quận

Dùng `pivot_table` (hoặc `groupby` + `value_counts(normalize=True)`) tính **tỷ trọng
3 phân khúc trong từng quận lớn**. Quận nào "cao cấp" nhất? Kết quả có khớp với bảng
xếp hạng trung vị của lab 2 (Lo Barnechea, Vitacura dẫn đầu) không?

In [ ]:
# Viết bài tự làm của bạn ở đây

## Tóm tắt buổi lab

| Bạn đã làm | Sẽ gặp lại ở |
|---|---|
| map với hàm sinh cột phân loại | mọi bước làm giàu dữ liệu của bài tập lớn |
| agg đặt tên nhiều cột — đọc bảng thành nhận xét | bảng KPI bài tập lớn |
| transform gắn thông tin nhóm về dòng (size, max) | QA outlier theo nhóm (buổi 10) |
| pivot cắt lớp một kết luận | kiểm chứng "kết luận sống sót qua cắt lớp" (buổi 14) |
| merge + validate + giải thích từng NaN | ghép bảng vùng/tỷ giá trong bài tập lớn |

Buổi lý thuyết tới: dữ liệu ngoài file CSV — API, SQL và DuckDB.